# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the [FAIR<sup>2</sup>](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Display basic metadata about the dataset
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"License: {dataset.metadata.license}")
print(f"Version: {dataset.metadata.version}")
print(f"Keywords: {', '.join(dataset.metadata.keywords)}")
print(f"Identifier: {dataset.metadata.identifier}")

## 2. Data Overview
Review available record sets, fields, and their `@id` fields for further analysis.

In [ ]:
# List available record sets and their fields by @id
print("Available record sets:")
recordset_ids = []
if hasattr(dataset.metadata, 'recordSets') and dataset.metadata.recordSets:
    for rs in dataset.metadata.recordSets:
        rid = rs["@id"]
        name = getattr(rs, 'name', rs.get('name', rid))
        print(f"- {name} (@id={rid})")
        recordset_ids.append(rid)
        print("  Fields:")
        if hasattr(rs, 'fields') and rs.fields:
            for f in rs.fields:
                fname = getattr(f, 'name', f.get('name', f['@id']))
                print(f"    - {fname} (@id={f['@id']}) (dataType={f.get('dataType', 'unknown')})")
else:
    # Try finding alternatives if .recordSets is missing
    meta_json = dataset.metadata.to_json()
    recordset_objs = meta_json.get('recordSet', [])
    if recordset_objs:
        for rs in recordset_objs:
            rid = rs["@id"] if isinstance(rs, dict) else rs
            name = rs.get('name', rid) if isinstance(rs, dict) else rid
            print(f"- {name} (@id={rid})")
            recordset_ids.append(rid)
    else:
        print("No record sets found in metadata.")

# If no recordSets found in Step 1, try mlcroissant's helper (for Fair^2 datasets, main table usually has @id='https://api.app.sen.science/frontiers/7154140/8875020f-242a-4f80-afd4-a4cbac8715f0/main-record-set')
if not recordset_ids:
    print("\nFor this FAIR2 dataset, let's try to infer the main table's @id.")
    main_rs_id = 'https://api.app.sen.science/frontiers/7154140/8875020f-242a-4f80-afd4-a4cbac8715f0/main-record-set'
    recordset_ids = [main_rs_id]
print(f"Using record sets: {recordset_ids}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# We'll use the record set ID inferred or listed above
record_sets = recordset_ids
dataframes = {}

# Load data from each record set
for record_set in record_sets:
    print(f"Loading record set: {record_set}")
    try:
        records = list(dataset.records(record_set=record_set))
        df = pd.DataFrame(records)
        print(f" Number of records: {len(df)}")
        print(f" Columns: {df.columns.tolist()}")
        dataframes[record_set] = df
    except Exception as e:
        print(f"Failed to load record set {record_set}: {e}")

# Choose the first record set for demonstration
main_record_set_id = record_sets[0]
main_df = dataframes[main_record_set_id]
print(f"\nFields/Columns for {main_record_set_id}:")
print(main_df.columns.tolist())
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

In [ ]:
# Identify numeric fields in the main recordset DataFrame
numeric_fields = main_df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numeric fields found: {numeric_fields}")

# If no numeric field detected, try to convert likely candidates
if not numeric_fields:
    # Try common field names in proteomics
    for guess in ['Coverage%', 'MW', 'Peptides', 'UniquePeptides', 'Abundance', 'PSMs', 'pI', 'Score', 'NormalizedAbundance']:
        if guess in main_df.columns:
            main_df[guess] = pd.to_numeric(main_df[guess], errors='coerce')
    numeric_fields = main_df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"Numeric fields after conversion: {numeric_fields}")

# Select a numeric field for demonstration
numeric_field = numeric_fields[0] if numeric_fields else None
if numeric_field:
    print(f"\nExample numeric field selected: {numeric_field}")
    
    threshold = main_df[numeric_field].quantile(0.75) if main_df[numeric_field].nunique()>20 else main_df[numeric_field].median()
    filtered_df = main_df[main_df[numeric_field] > threshold]
    print(f"Filtered records where {numeric_field} > {threshold} (Top 25%): {len(filtered_df)} records")

    # Normalize the selected numeric field (Z-score normalization)
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(filtered_df[[numeric_field, norm_col]].head())

    # Group by a categorical column if present (e.g., 'Sample' or 'Description')
    potential_group_fields = [c for c in main_df.columns if main_df[c].dtype == "object" and c != numeric_field]
    print(f"Available fields for grouping: {potential_group_fields}")
    if potential_group_fields:
        group_field = potential_group_fields[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        grouped_df = grouped_df.rename(columns={numeric_field: f"mean_{numeric_field}"})
        print(f"\nMean {numeric_field} grouped by {group_field}:")
        print(grouped_df.head())
else:
    print("No numeric fields found for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Visualize distribution of the numeric field
if numeric_field:
    plt.figure(figsize=(7,4))
    main_df[numeric_field].hist(bins=30, alpha=0.7, color='skyblue')
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.title(f"Distribution of {numeric_field}")
    plt.show()
    
    # Boxplot
    plt.figure(figsize=(4,5))
    main_df.boxplot(column=numeric_field)
    plt.title(f"Boxplot of {numeric_field}")
    plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook we loaded the FAIR<sup>2</sup> dataset on human extracellular vesicle proteins using the Croissant schema and performed basic exploratory analysis via `mlcroissant`.

- We retrieved available record sets and fields by their `@id` values.
- Converted and explored main numeric fields (e.g. protein abundance, MW, or coverage%).
- Demonstrated filtering, normalization, and grouping.
- Visualized main quantitative properties and discussed possible groupings.

This workflow can be extended for downstream biological or machine-learning analyses as required.